This notebook is to train the polygon segmentation model 

In [1]:
%%capture
!pip install segmentation-models-pytorch pytorch-lightning

In [2]:
import os
import json
import re
import random
import time
from pathlib import Path
from dataclasses import dataclass
from datetime import datetime, timedelta

import numpy as np
import pandas as pd

import cv2
from PIL import Image, ImageDraw
from skimage.measure import label, regionprops

import albumentations as A
from albumentations.pytorch import ToTensorV2

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import pytorch_lightning as pl
import segmentation_models_pytorch as smp

import matplotlib.pyplot as plt
from tqdm import tqdm

from shapely.geometry import Polygon

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

In [3]:
!mkdir -p /kaggle/working/images
!cp -r /kaggle/input/barbadosoriginalall/survey_plans/* /kaggle/working/images/

In [4]:
def set_global_seed(seed=1056):
    """Set seed for reproducibility across all libraries"""
    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

    try:
        import pytorch_lightning as pl
        pl.seed_everything(seed, workers=True)
    except ImportError:
        pass

    print(f"Global seed set to {seed}")

In [5]:
@dataclass
class CFG:
    # Data paths
    DATA_ROOT: str = "/kaggle/input/barbadosoriginalall"
    IMAGES_DIR: str = "/kaggle/working/images/train"
    COCO_JSON: str = "/kaggle/input/barbadosoriginalall/result.json"
    OUT_DIR: str = "/kaggle/working"

    # Model configuration - Enhanced for better accuracy
    ENCODER: str = "efficientnet-b7"  # Upgraded encoder
    ARCHITECTURE: str = "UnetPlusPlus"  # Better for precise boundaries
    FULL_SIZE_TRAIN: int = 1024  # Optimized for L4 GPU memory

    # Data splits
    VAL_FRACTION: float = 0.15
    TEST_FRACTION: float = 0.10
    SEED: int = 42

    # Training parameters - L4 GPU optimized
    BATCH: int = 1  # Optimized for L4
    EPOCHS: int = 100
    LR: float = 1e-4  # More conservative for stability
    NUM_WORKERS: int = 4
    AMP: bool = True
    EARLY_STOP: int = 10

    # Enhanced loss configuration
    USE_BOUNDARY_LOSS: bool = True
    USE_FOCAL_LOSS: bool = True
    BOUNDARY_WEIGHT: float = 0.4
    FOCAL_ALPHA: float = 0.25
    FOCAL_GAMMA: float = 2.0

    # Enhanced inference configuration
    INFERENCE_THR: float = 0.3  # Lower threshold for better recall
    USE_TTA: bool = True
    USE_CRF: bool = False

    # Enhanced post-processing
    MORPH_KERNEL_SIZE: int = 3
    GAUSSIAN_KERNEL_SIZE: int = 3
    CORNER_DETECTION: bool = True
    MIN_CONTOUR_AREA: int = 5000

CFG = CFG()
Path(CFG.OUT_DIR).mkdir(parents=True, exist_ok=True)

# Ensure reproducibility with the config seed
set_global_seed(CFG.SEED)
print(CFG)

Seed set to 42


Global seed set to 42
CFG(DATA_ROOT='/kaggle/input/barbadosoriginalall', IMAGES_DIR='/kaggle/working/images/train', COCO_JSON='/kaggle/input/barbadosoriginalall/result.json', OUT_DIR='/kaggle/working', ENCODER='efficientnet-b7', ARCHITECTURE='UnetPlusPlus', FULL_SIZE_TRAIN=1024, VAL_FRACTION=0.15, TEST_FRACTION=0.1, SEED=42, BATCH=1, EPOCHS=100, LR=0.0001, NUM_WORKERS=4, AMP=True, EARLY_STOP=10, USE_BOUNDARY_LOSS=True, USE_FOCAL_LOSS=True, BOUNDARY_WEIGHT=0.4, FOCAL_ALPHA=0.25, FOCAL_GAMMA=2.0, INFERENCE_THR=0.3, USE_TTA=True, USE_CRF=False, MORPH_KERNEL_SIZE=3, GAUSSIAN_KERNEL_SIZE=3, CORNER_DETECTION=True, MIN_CONTOUR_AREA=5000)


In [6]:
DATA_PATH = "/kaggle/input/barbadosoriginalall"
EXTRACT_PATH = "/kaggle/working/images"

import pandas as pd

def load_datasets():
    """Load and return train, test, and sample submission datasets"""
    train_df = pd.read_csv(DATA_PATH + '/Train.csv')

    print(f"Train dataset shape: {train_df.shape}")

    return train_df

train_df = load_datasets()
display(train_df.head())

Train dataset shape: (966, 10)


,ID,Land Surveyor,Surveyed For,Certified date,Total Area,Unit of Measurement,Address,Parish,LT Num,geometry
0,7703-079,Jennifer Alleyne,Rico R Wiltshire,2020-09-17,596.0,sq m,"Lot 468 Ruby Development Stage 3, Part 1",St. Philip,77.03.09.068,POLYGON Z ((42496.26640000008 69380.3217999991...
1,7703-062,Ronald Greene,Sylvia Wilson,2018-10-01,475.0,sq m,Lot 588 Ruby Development Stage 3 Part 1,St. Philip,77.03.07.049,POLYGON Z ((42615.50050000008 69154.0432999990...
2,7704-115,Ronald Greene,Frank Ivy,2017-07-06,8432.7,sq m,Belair,St. Philip,77.04.01.013,POLYGON Z ((42905.95720000006 68481.6059000007...
3,7704-116,Ronald Greene,Anderson Jones,2018-04-02,2112.6,sq m,Lot 1 Belair,St. Philip,77.04.01.013,POLYGON Z ((42905.95720000006 68481.6059000007...
4,8603-121,Michelle St.Clair,McCollin Hendy,2023-10-11,598.4,sq m,Kirtons,St. Philip,86.03.01.019,POLYGON Z ((39989.26090000011 66736.3342000003...


In [7]:
import zipfile
import os

def organize_images():
    """Extract images from zip and organize into train/test folders"""

    # Count total images
    total_images = len([f for f in os.listdir(EXTRACT_PATH) if f.endswith('.jpg')])
    print(f"Total images found: {total_images}")

    def create_train_data(df, data_type):
        data_dir = os.path.join(EXTRACT_PATH, data_type)
        os.makedirs(data_dir, exist_ok=True)

        for _, row in df.iterrows():
            id_value = row['ID']
            image_name = f'anonymised_{id_value}.jpg'
            source_path = os.path.join(EXTRACT_PATH, image_name)

            if os.path.exists(source_path):
                dest_path = os.path.join(data_dir, image_name)
                os.rename(source_path, dest_path)
    create_train_data(train_df, 'train')

organize_images()

Total images found: 877


In [8]:
images_dir = Path(CFG.IMAGES_DIR)
masks_dir = Path(CFG.OUT_DIR) / "masks_full_json"
masks_dir.mkdir(parents=True, exist_ok=True)

def safe_stem(p):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", Path(p).stem)

rows = []
try:
    with open(CFG.COCO_JSON, "r") as f:
        data = json.load(f)
    images = {img["id"]: img for img in data["images"]}
    by_img = {}
    for a in data["annotations"]:
        by_img.setdefault(a["image_id"], []).append(a)

    for img_id, info in tqdm(images.items()):
        W, H = info["width"], info["height"]
        segs = by_img.get(img_id, [])
        mask = Image.new("L", (W, H), 0)
        d = ImageDraw.Draw(mask)
        for a in segs:
            seg = a.get("segmentation", [])
            if isinstance(seg, list):
                for poly in seg:
                    if len(poly) >= 6:
                        pts = [(poly[i], poly[i+1]) for i in range(0, len(poly), 2)]
                        d.polygon(pts, fill=255)
        stem = safe_stem(info["file_name"])
        out_path = masks_dir / f"{stem}.png"
        mask.save(out_path)
        rows.append({"basename": stem,
                     "image_path": str(images_dir / Path(info["file_name"]).name),
                     "mask_path": str(out_path),
                     "width": W, "height": H})
    manifest_full = Path(CFG.OUT_DIR) / "manifest_full_images.csv"
    pd.DataFrame(rows).to_csv(manifest_full, index=False)
    print("Wrote:", manifest_full, "rows:", len(rows))
except FileNotFoundError:
    print("COCO JSON not found; skip this cell if using YOLO labels.")

100%|██████████| 676/676 [00:51<00:00, 13.22it/s]

Wrote: /kaggle/working/manifest_full_images.csv rows: 676


In [9]:
df = pd.read_csv("/kaggle/working/manifest_full_images.csv")

# Function to strip everything before 'anonymised_'
def fix_image_path(path):
    import re
    match = re.search(r"anonymised_.*", path)
    if match:
        return f"/kaggle/working/images/train/{match.group(0)}"
    return path  # fallback if no match

# Apply to column
df["image_path"] = df["image_path"].apply(fix_image_path)

df.to_csv("/kaggle/working/manifest_full_images.csv", index=False)

print(df["image_path"].head())

0    /kaggle/working/images/train/anonymised_4516-0...
1    /kaggle/working/images/train/anonymised_4516-0...
2    /kaggle/working/images/train/anonymised_4516-0...
3    /kaggle/working/images/train/anonymised_4516-0...
4    /kaggle/working/images/train/anonymised_4516-0...
Name: image_path, dtype: object


In [10]:
rng = np.random.RandomState(CFG.SEED)
df_full = pd.read_csv(Path(CFG.OUT_DIR) / "manifest_full_images.csv").sample(frac=1.0, random_state=CFG.SEED)
n = len(df_full)
n_test = int(n * CFG.TEST_FRACTION)
n_val = int(n * CFG.VAL_FRACTION)

test_full = df_full.iloc[:n_test].copy()
val_full = df_full.iloc[n_test:n_test+n_val].copy()
train_full = df_full.copy() # Use full data for training

def fix_paths(df, images_dir, masks_dir):
    df["image_path"] = df["image_path"].str.replace(r".*[/\\]+upload[/\\]+5[/\\]+", str(images_dir) + "/", regex=True)
    df["mask_path"] = df["mask_path"].str.replace(r".*masks_full_json[/\\]+", str(masks_dir) + "/", regex=True)
    return df

train_full = fix_paths(train_full, CFG.IMAGES_DIR, str(Path(CFG.OUT_DIR)/"masks_full_json"))
val_full = fix_paths(val_full, CFG.IMAGES_DIR, str(Path(CFG.OUT_DIR)/"masks_full_json"))
test_full = fix_paths(test_full, CFG.IMAGES_DIR, str(Path(CFG.OUT_DIR)/"masks_full_json"))

train_full.to_csv(Path(CFG.OUT_DIR)/"train_full.csv", index=False)
val_full.to_csv(Path(CFG.OUT_DIR)/"val_full.csv", index=False)
test_full.to_csv(Path(CFG.OUT_DIR)/"test_full.csv", index=False)

print("Splits done with seed:", CFG.SEED)

Splits done with seed: 42


In [11]:
# Data augmentation
PREPROC = smp.encoders.get_preprocessing_params(CFG.ENCODER, pretrained="imagenet")
IM_MEAN, IM_STD = PREPROC["mean"], PREPROC["std"]

def get_enhanced_train_aug(size=None, seed=None):
    """Enhanced augmentations optimized for document boundary detection"""
    if seed is None:
        seed = CFG.SEED

    ops = []
    if size:
        ops.append(A.Resize(size, size, interpolation=cv2.INTER_LINEAR))

    # Document-specific augmentations
    ops += [
        # Geometric transformations that preserve document structure
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.3),
        A.RandomRotate90(p=0.5),

        # Slight perspective changes (common in document scanning)
        A.OneOf([
            A.ShiftScaleRotate(
                shift_limit=0.05,
                scale_limit=0.1,
                rotate_limit=3,
                border_mode=cv2.BORDER_CONSTANT,
                value=0,
                mask_value=0,
                p=1.0
            ),
            A.ElasticTransform(
                alpha=1,
                sigma=20,
                alpha_affine=10,
                border_mode=cv2.BORDER_CONSTANT,
                value=0,
                mask_value=0,
                p=1.0
            ),
        ], p=0.3),

        # Enhanced color/lighting augmentations for document scans
        A.OneOf([
            A.RandomBrightnessContrast(
                brightness_limit=0.15,
                contrast_limit=0.15,
                p=1.0
            ),
            A.CLAHE(clip_limit=3.0, tile_grid_size=(4, 4), p=1.0),
            A.RandomGamma(gamma_limit=(85, 115), p=1.0),
        ], p=0.6),

        # Color variations
        A.HueSaturationValue(
            hue_shift_limit=8,
            sat_shift_limit=12,
            val_shift_limit=8,
            p=0.3
        ),

        # Document-specific noise and artifacts
        A.OneOf([
            A.GaussNoise(var_limit=(3.0, 12.0), p=1.0),
            A.ISONoise(color_shift=(0.01, 0.05), intensity=(0.1, 0.5), p=1.0),
            A.MultiplicativeNoise(multiplier=[0.98, 1.02], p=1.0),
        ], p=0.25),

        # Slight blur to simulate camera focus variations
        A.OneOf([
            A.MotionBlur(blur_limit=3, p=1.0),
            A.GaussianBlur(blur_limit=3, p=1.0),
        ], p=0.15),

        # Normalize and convert to tensor
        A.Normalize(mean=IM_MEAN, std=IM_STD),
        ToTensorV2(),
    ]
    return A.Compose(ops, additional_targets={}, seed=seed)

def get_val_aug(size=None):
    ops = []
    if size:
        ops.append(A.Resize(size, size, interpolation=cv2.INTER_LINEAR))
    ops += [A.Normalize(mean=IM_MEAN, std=IM_STD), ToTensorV2()]
    return A.Compose(ops)

class SegDS(Dataset):
    def __init__(self, csv_path, size=None, train=True, seed=None):
        self.df = pd.read_csv(csv_path)
        self.seed = seed if seed is not None else CFG.SEED
        self.tr = get_enhanced_train_aug(size, self.seed) if train else get_val_aug(size)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = cv2.imread(row["image_path"], cv2.IMREAD_COLOR)[:,:,::-1]
        mask = cv2.imread(row["mask_path"], cv2.IMREAD_GRAYSCALE)
        mask = (mask > 127).astype(np.uint8)

        # Apply augmentation
        aug = self.tr(image=img, mask=mask)
        img, mask = aug["image"], aug["mask"].unsqueeze(0).float()
        return img, mask

config.json:   0%|          | 0.00/106 [00:00<?, ?B/s]

In [12]:
def build_model():
    """Build enhanced model for better boundary detection"""
    return smp.UnetPlusPlus(
        encoder_name=CFG.ENCODER,
        encoder_weights="imagenet",
        in_channels=3,
        classes=1,
        decoder_attention_type="scse",
        decoder_channels=(512, 256, 128, 64, 32),
        decoder_use_batchnorm=True,
    )

class FocalLoss(torch.nn.Module):
    """Focal Loss for addressing class imbalance"""
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        bce_loss = F.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
        pt = torch.exp(-bce_loss)
        focal_loss = self.alpha * (1-pt)**self.gamma * bce_loss
        return focal_loss.mean()

class EnhancedLoss(torch.nn.Module):
    """Multi-component loss for precise boundary detection"""
    def __init__(self, focal_weight=0.3, dice_weight=0.4, boundary_weight=0.3):
        super().__init__()
        self.focal_weight = focal_weight
        self.dice_weight = dice_weight
        self.boundary_weight = boundary_weight
        self.focal_loss = FocalLoss(alpha=CFG.FOCAL_ALPHA, gamma=CFG.FOCAL_GAMMA)

    def forward(self, logits, targets):
        # Focal loss for class imbalance
        focal = self.focal_loss(logits, targets)

        # Dice loss for overlap
        probs = torch.sigmoid(logits)
        num = 2*(probs*targets).sum(dim=(1,2,3)) + 1e-6
        den = probs.sum(dim=(1,2,3)) + targets.sum(dim=(1,2,3)) + 1e-6
        dice_loss = 1 - (num/den).mean()

        # Enhanced boundary loss using Sobel edge detection
        boundary_loss = 0.0
        if self.boundary_weight > 0:
            pred_edges = self.get_edges_sobel(probs)
            target_edges = self.get_edges_sobel(targets)
            boundary_loss = F.mse_loss(pred_edges, target_edges)

        total_loss = (self.focal_weight * focal +
                     self.dice_weight * dice_loss +
                     self.boundary_weight * boundary_loss)

        return total_loss

    def get_edges_sobel(self, masks):
        """Extract edges using Sobel operator"""
        # Sobel X
        sobel_x = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]],
                              dtype=torch.float32).view(1,1,3,3).to(masks.device)
        # Sobel Y
        sobel_y = torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]],
                              dtype=torch.float32).view(1,1,3,3).to(masks.device)

        edge_x = F.conv2d(masks, sobel_x, padding=1)
        edge_y = F.conv2d(masks, sobel_y, padding=1)
        edges = torch.sqrt(edge_x**2 + edge_y**2 + 1e-6)
        return edges

def dice_coeff(pred, target, eps=1e-6):
    pred = (pred>0.5).float()
    inter = (pred*target).sum(dim=(1,2,3))
    den = pred.sum(dim=(1,2,3)) + target.sum(dim=(1,2,3)) + eps
    return (2*inter+eps)/den

class LitSeg(pl.LightningModule):
    def __init__(self, size=None, lr=CFG.LR, pos_weight=None):
        super().__init__()
        self.save_hyperparameters()
        self.model = build_model()

        # Enhanced loss function
        if CFG.USE_FOCAL_LOSS and CFG.USE_BOUNDARY_LOSS:
            self.crit = EnhancedLoss()
        elif CFG.USE_FOCAL_LOSS:
            self.crit = FocalLoss(alpha=CFG.FOCAL_ALPHA, gamma=CFG.FOCAL_GAMMA)
        else:
            self.crit = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)

        self.lr = lr

    def forward(self, x):
        return self.model(x)

    def configure_optimizers(self):
        opt = torch.optim.AdamW(
            self.parameters(),
            lr=self.lr,
            weight_decay=1e-4,
            eps=1e-8
        )

        scheduler = torch.optim.lr_scheduler.OneCycleLR(
            opt,
            max_lr=self.lr,
            total_steps=self.trainer.estimated_stepping_batches,
            pct_start=0.1,
            anneal_strategy="cos",
        )

        return {
            "optimizer": opt,
            "lr_scheduler": {
                "scheduler": scheduler,
                "interval": "step"
            }
        }

    def _step(self, batch, stage):
        imgs, masks = batch
        logits = self(imgs)
        loss = self.crit(logits, masks)
        probs = torch.sigmoid(logits)
        dice = dice_coeff(probs, masks).mean()
        self.log(f"{stage}_loss", loss, prog_bar=True, on_step=False, on_epoch=True)
        self.log(f"{stage}_dice", dice, prog_bar=True, on_step=False, on_epoch=True)
        return loss

    def training_step(self, batch, _):
        return self._step(batch, "train")

    def validation_step(self, batch, _):
        return self._step(batch, "val")

def train_run(train_csv, val_csv, size, tag, pos_weight=None):
    # Set seed for worker processes
    def worker_init_fn(worker_id):
        np.random.seed(CFG.SEED + worker_id)
        random.seed(CFG.SEED + worker_id)

    train_ds = SegDS(train_csv, size=size, train=True, seed=CFG.SEED)
    val_ds = SegDS(val_csv, size=size, train=False, seed=CFG.SEED)

    train_dl = DataLoader(train_ds,
                          batch_size=CFG.BATCH,
                          shuffle=True,
                          num_workers=CFG.NUM_WORKERS,
                          pin_memory=True,
                          worker_init_fn=worker_init_fn,
                          generator=torch.Generator().manual_seed(CFG.SEED))
    val_dl = DataLoader(val_ds,
                        batch_size=CFG.BATCH,
                        shuffle=False,
                        num_workers=CFG.NUM_WORKERS,
                        pin_memory=True,
                        worker_init_fn=worker_init_fn)

    ckpt_dir = Path(CFG.OUT_DIR)/"checkpoints"
    ckpt_dir.mkdir(parents=True, exist_ok=True)

    callbacks = [
        pl.callbacks.ModelCheckpoint(
            monitor="val_dice", mode="max", save_top_k=1,
            dirpath=str(ckpt_dir), filename=f"{tag}-best"
        ),
        pl.callbacks.EarlyStopping(
            monitor="val_dice", mode="max", patience=CFG.EARLY_STOP
        ),
        pl.callbacks.LearningRateMonitor()
    ]

    trainer = pl.Trainer(
        max_epochs=CFG.EPOCHS,
        precision="16-mixed" if CFG.AMP else 32,
        callbacks=callbacks,
        log_every_n_steps=20,
        deterministic=True,
        enable_checkpointing=True
    )

    model = LitSeg(size=size, lr=CFG.LR, pos_weight=pos_weight)
    trainer.fit(model, train_dl, val_dl)
    return str(callbacks[0].best_model_path)

In [13]:
df = pd.read_csv(Path(CFG.OUT_DIR) / "train_full.csv")

p = df.iloc[0]["image_path"]
print("Path:", p)
print("Exists:", Path(p).exists())

Path: /kaggle/working/images/train/anonymised_BYLS-090.jpg
Exists: True


In [16]:
set_global_seed(CFG.SEED)

def estimate_pos_weight(csv_path):
    df = pd.read_csv(csv_path)
    pos, total = 0, 0
    for _,r in df.iterrows():
        m = (cv2.imread(r["mask_path"], cv2.IMREAD_GRAYSCALE)>127).astype(np.uint8)
        pos += m.sum()
        total += m.size
    neg = total - pos
    if pos == 0:
        return None
    return torch.tensor([max(1.0, neg/pos)], dtype=torch.float32)

print("Estimating pos_weight...")
pos_w_full = estimate_pos_weight(Path(CFG.OUT_DIR)/"train_full.csv")
print(f"pos_weight: {pos_w_full}")

print(f"\nStarting training at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Using seed: {CFG.SEED}")
training_start = time.time()

ckpt_full = train_run(
    Path(CFG.OUT_DIR)/"train_full.csv",
    Path(CFG.OUT_DIR)/"val_full.csv",
    size=CFG.FULL_SIZE_TRAIN,
    tag="enhanced_boundary",
    pos_weight=pos_w_full
)

training_duration = time.time() - training_start
print(f"Training completed in {str(timedelta(seconds=int(training_duration)))}")
print("Best checkpoint:", ckpt_full)

Seed set to 42


Global seed set to 42
Estimating pos_weight...


/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
/tmp/ipykernel_55/221255848.py:24: UserWarning: Argument(s) 'value, mask_value' are not valid for transform ShiftScaleRotate
  A.ShiftScaleRotate(
/tmp/ipykernel_55/221255848.py:33: UserWarning: Argument(s) 'alpha_affine, value, mask_value' are not valid for transform ElasticTransform
  A.ElasticTransform(
/tmp/ipykernel_55/221255848.py:65: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(3.0, 12.0), p=1.0),
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores


pos_weight: tensor([13.2830])

Starting training at 2025-12-19 01:05:47
Using seed: 42


model.safetensors:   0%|          | 0.00/267M [00:00<?, ?B/s]

2025-12-19 01:05:54.187039: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1766106354.370119      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1766106354.430693      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1766106354.881738      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766106354.881788      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766106354.881791      55 computation_placer.cc:177] computation placer alr

┏━━━┳━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type         ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model │ UnetPlusPlus │ 74.3 M │ train │     0 │
│ 1 │ crit  │ EnhancedLoss │      0 │ train │     0 │
└───┴───────┴──────────────┴────────┴───────┴───────┘

Trainable params: 74.3 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 74.3 M                                                                                               
Total estimated model params size (MB): 297                                                                        
Modules in train mode: 1184                                                                                        
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

Training completed in 2:56:23
Best checkpoint: /kaggle/working/checkpoints/enhanced_boundary-best.ckpt


In [22]:
!zip -r checkpoints.zip /kaggle/working/checkpoints/

  adding: kaggle/working/checkpoints/ (stored 0%)
  adding: kaggle/working/checkpoints/enhanced_boundary-best.ckpt (deflated 8%)


Check if output is correct

In [18]:
df = pd.read_csv("/kaggle/working/manifest_full_images.csv")

In [19]:
df["image_path"]
value = df.at[0, "image_path"]   # row 0, column 'image_path'
p = Path(value)
print(p.resolve())

/kaggle/working/images/train/anonymised_4516-018.jpg


In [20]:
from pathlib import Path

folder = Path("/kaggle/working/images")

num_files_top = sum(1 for f in folder.glob("*") if f.is_file())
print("Top-level files:", num_files_top)

Top-level files: 219
